# 08 - Train intent


In [8]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import Pipeline

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

TRAIN_PATH = PROJECT_ROOT / "intent_data" / "train.csv"
TEST_PATH = PROJECT_ROOT / "intent_data" / "test.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "intent_clf.joblib"
REPORT_DIR = PROJECT_ROOT / "eval" / "reports"

from app.chatbot.intent.labels import INTENT_LABELS
from app.chatbot.intent.classifier import IntentClassifier

print("PROJECT_ROOT", PROJECT_ROOT)
print("labels", INTENT_LABELS)

PROJECT_ROOT d:\TTTN-VaxCare-2026\vaxcare-ai
labels ['faq_vaccine', 'schedule_advice', 'history_lookup', 'vaxcare_lookup', 'chitchat', 'out_of_scope']


In [9]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("train", len(train_df), "test", len(test_df))
print(train_df["label"].value_counts())

train 406 test 76
label
faq_vaccine        166
vaxcare_lookup      95
schedule_advice     71
history_lookup      54
out_of_scope        10
chitchat            10
Name: count, dtype: int64


In [10]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs",
        random_state=42,
    )),
])

X_train = train_df["text"].astype(str)
y_train = train_df["label"].astype(str)
pipe.fit(X_train, y_train)
print("classes_", list(pipe.classes_))

classes_ ['chitchat', 'faq_vaccine', 'history_lookup', 'out_of_scope', 'schedule_advice', 'vaxcare_lookup']


c:\Users\Bao Quan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


In [11]:
X_test = test_df["text"].astype(str)
y_test = test_df["label"].astype(str)
y_pred = pipe.predict(X_test)

labels = list(INTENT_LABELS)
acc = accuracy_score(y_test, y_pred)
f1_mac = f1_score(y_test, y_pred, average="macro", labels=labels, zero_division=0)
f1_w = f1_score(y_test, y_pred, average="weighted", labels=labels, zero_division=0)
print(f"accuracy={acc:.4f}  f1_macro={f1_mac:.4f}  f1_weighted={f1_w:.4f}")
print(classification_report(y_test, y_pred, labels=labels, zero_division=0))

accuracy=0.9079  f1_macro=0.8250  f1_weighted=0.8992
                 precision    recall  f1-score   support

    faq_vaccine       0.94      1.00      0.97        29
schedule_advice       0.80      1.00      0.89        12
 history_lookup       1.00      0.78      0.88         9
 vaxcare_lookup       0.94      1.00      0.97        16
       chitchat       1.00      0.60      0.75         5
   out_of_scope       0.67      0.40      0.50         5

       accuracy                           0.91        76
      macro avg       0.89      0.80      0.83        76
   weighted avg       0.91      0.91      0.90        76



In [12]:
# Ngưỡng confidence trên tập test
proba = pipe.predict_proba(X_test)
max_conf = proba.max(axis=1)
print("conf mean", float(max_conf.mean()), "p10", float(np.percentile(max_conf, 10)), "p50", float(np.median(max_conf)))

for thr in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    mask = max_conf >= thr
    if mask.sum() == 0:
        continue
    acc_t = accuracy_score(y_test[mask], y_pred[mask])
    coverage = float(mask.mean())
    print(f"thr={thr:.2f}  coverage={coverage:.2f}  acc_on_confident={acc_t:.4f}")

conf mean 0.5843291415606235 p10 0.36632772435276306 p50 0.6034075716588725
thr=0.35  coverage=0.91  acc_on_confident=0.9855
thr=0.40  coverage=0.87  acc_on_confident=1.0000
thr=0.45  coverage=0.80  acc_on_confident=1.0000
thr=0.50  coverage=0.79  acc_on_confident=1.0000
thr=0.55  coverage=0.71  acc_on_confident=1.0000
thr=0.60  coverage=0.54  acc_on_confident=1.0000


In [13]:
# Chọn ngưỡng mặc định (có thể chỉnh sau khi xem bảng trên)
MIN_CONF = 0.45

clf = IntentClassifier(
    pipeline=pipe,
    labels=list(pipe.classes_),
    min_confidence=MIN_CONF,
)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
clf.save(MODEL_PATH)
print("saved", MODEL_PATH)

# Smoke
for q in [
    "Vaccine HPV phòng bệnh gì?",
    "Tôi 20 tuổi nên tiêm gì?",
    "Xin chào",
    "Đặt lịch tiêm MMR",
    "Tôi bị đau ngực khó thở",
]:
    lab, conf = clf.predict(q)
    print(f"{conf:.2f}  {lab:20s}  | {q}")

saved d:\TTTN-VaxCare-2026\vaxcare-ai\models\intent_clf.joblib
0.56  faq_vaccine           | Vaccine HPV phòng bệnh gì?
0.62  schedule_advice       | Tôi 20 tuổi nên tiêm gì?
0.38  chitchat              | Xin chào
0.64  vaxcare_lookup        | Đặt lịch tiêm MMR
0.47  out_of_scope          | Tôi bị đau ngực khó thở


In [14]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
report = {
    "accuracy": float(acc),
    "f1_macro": float(f1_mac),
    "f1_weighted": float(f1_w),
    "min_confidence": MIN_CONF,
    "n_train": len(train_df),
    "n_test": len(test_df),
    "model": "tfidf+logreg",
}
out = REPORT_DIR / "intent_metrics.json"
out.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("report", out)

report d:\TTTN-VaxCare-2026\vaxcare-ai\eval\reports\intent_metrics.json
